In [1]:
import dotenv
import wandb
from src.api.run.xgboost import sweep_xgboost_ensemble
from src.api.sweep import wandb_sweep

D:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statemen

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: schurtenberger-david (david-schurtenberger) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [14]:
max_runs = 100
sweep_config = {
    "name": "XGBoost Ensemble",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "xgboost_config": {
            "parameters": {
                "num_rounds": {"values": [600, 700, 800]},
                "num_parallel_tree": {"values": [2, 3, 4]},
                "min_child_weight": {"values": [1, 2, 4, 8]},
                "learning_rate": {"distribution": "uniform", "min": 1e-4, "max": 1e-1},
                "gamma": {"distribution": "uniform", "min": 0.0, "max": 0.3},
                "max_depth": {"distribution": "int_uniform", "min": 2, "max": 8},
                "subsample": {"values": [0.8, 0.7, 0.6, 0.5]},
                "colsample_bynode": {"values": [1.0, 0.9, 0.8, 0.7, 0.6]},
                "reg_lambda": {"distribution": "uniform", "min": 0.5, "max": 3.0},
                "reg_alpha": {"distribution": "uniform", "min": 0.0, "max": 1.5},
                "max_bin": {"distribution": "int_uniform", "min": 8, "max": 128},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 85},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_xgboost_ensemble, run_count=max_runs, project="xgboost-regressor")

Create sweep with ID: kqhcxnn2
Sweep URL: https://wandb.ai/aicomp-mmlm/xgboost-regressor/sweeps/kqhcxnn2


wandb: Agent Starting Run: 4gsoq03a with config:
wandb: 	run_config: {'data_loader': 'season_average_ensemble', 'num_features': 62, 'start_season': 2003, 'valid_season': 2025}
wandb: 	xgboost_config: {'colsample_bynode': 0.6, 'gamma': 0.05447910223184967, 'learning_rate': 0.01216229418162542, 'max_bin': 31, 'max_depth': 3, 'min_child_weight': 8, 'num_parallel_tree': 4, 'num_rounds': 600, 'reg_alpha': 0.4291973702789944, 'reg_lambda': 1.858947739737627, 'subsample': 0.5}


### Submission from best Model

In [3]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.xgboost import EnsembleXGBRegressorModel, XGBHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import create_submission

In [4]:
run = wandb.Api().run("aicomp-mmlm/xgboost-regressor/repxxb8d")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 14,
  'start_season': 2003,
  'valid_season': 2025},
 'xgboost_config': {'gamma': 0.055374053199364,
  'max_bin': 115,
  'max_depth': 2,
  'reg_alpha': 0.5143332290078902,
  'subsample': 0.7,
  'num_rounds': 800,
  'reg_lambda': 1.1176227175094051,
  'learning_rate': 0.006483532039905739,
  'colsample_bynode': 0.6,
  'min_child_weight': 8,
  'num_parallel_tree': 3}}

In [5]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=14, valid_season=2025, start_season=2003, data_loader='season_average_ensemble')

In [6]:
hyperparameters = XGBHyperparamConfig(**config.get("xgboost_config", {}))
hyperparameters

XGBHyperparamConfig(num_rounds=800, device='cpu', objective='reg:squarederror', booster='gbtree', learning_rate=0.006483532039905739, gamma=0.055374053199364, max_depth=2, min_child_weight=8, num_parallel_tree=3, max_delta_step=0, subsample=0.7, colsample_bytree=0.8, colsample_bylevel=1.0, colsample_bynode=0.6, reg_lambda=1.1176227175094051, reg_alpha=0.5143332290078902, tree_method='hist', max_bin=115, grow_policy='lossguide', seed=42)

In [7]:
model = EnsembleXGBRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [8]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_xgboost_ensemble_{season}.csv", fit=True)

metrics: {'train_brier_ensemble': np.float64(0.16099961903161794)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.18087972269115138)}, step: 2003
metrics: {'train_brier_ensemble': np.float64(0.16104044940971765)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.17577865723266775)}, step: 2004
metrics: {'train_brier_ensemble': np.float64(0.16105613004977298)}, step: 2005
metrics: {'valid_brier_ensemble': np.float64(0.17677344956827715)}, step: 2005
metrics: {'train_brier_ensemble': np.float64(0.16056273118783454)}, step: 2006
metrics: {'valid_brier_ensemble': np.float64(0.19595125462144147)}, step: 2006
metrics: {'train_brier_ensemble': np.float64(0.16153205142687302)}, step: 2007
metrics: {'valid_brier_ensemble': np.float64(0.15749700220273932)}, step: 2007
metrics: {'train_brier_ensemble': np.float64(0.16178436231768134)}, step: 2008
metrics: {'valid_brier_ensemble': np.float64(0.15453598232635324)}, step: 2008
metrics: {'train_brier_ensemble': np.float64(0.16144

WindowsPath('D:/git/code/submissions/submission_xgboost_ensemble_2025.csv')

### Submission from best Model (Default Features)

In [9]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.xgboost import EnsembleXGBRegressorModel, XGBHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import create_submission

In [10]:
run = wandb.Api().run("aicomp-mmlm/xgboost-regressor/kvq12vlj")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 0,
  'start_season': 2003,
  'valid_season': 2025},
 'xgboost_config': {'gamma': 0.2713918119977201,
  'max_bin': 10,
  'max_depth': 3,
  'reg_alpha': 0.14729887618819598,
  'subsample': 0.8,
  'num_rounds': 1000,
  'reg_lambda': 2.929595942654601,
  'learning_rate': 0.004496429935050683,
  'colsample_bynode': 0.9,
  'min_child_weight': 2,
  'num_parallel_tree': 2}}

In [11]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=0, valid_season=2025, start_season=2003, data_loader='season_average_ensemble')

In [12]:
hyperparameters = XGBHyperparamConfig(**config.get("xgboost_config", {}))
hyperparameters

XGBHyperparamConfig(num_rounds=1000, device='cpu', objective='reg:squarederror', booster='gbtree', learning_rate=0.004496429935050683, gamma=0.2713918119977201, max_depth=3, min_child_weight=2, num_parallel_tree=2, max_delta_step=0, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, colsample_bynode=0.9, reg_lambda=2.929595942654601, reg_alpha=0.14729887618819598, tree_method='hist', max_bin=10, grow_policy='lossguide', seed=42)

In [13]:
model = EnsembleXGBRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [14]:
season = 2025
create_submission(
    season=season, model=model, filename=f"submission_xgboost_ensemble_default_features_{season}.csv", fit=True
)

metrics: {'train_brier_ensemble': np.float64(0.1565200488070149)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.1862835541961097)}, step: 2003
metrics: {'train_brier_ensemble': np.float64(0.15702091831941922)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.173461007357815)}, step: 2004
metrics: {'train_brier_ensemble': np.float64(0.156971100376613)}, step: 2005
metrics: {'valid_brier_ensemble': np.float64(0.16628978860836682)}, step: 2005
metrics: {'train_brier_ensemble': np.float64(0.1562914691408412)}, step: 2006
metrics: {'valid_brier_ensemble': np.float64(0.19733562775679758)}, step: 2006
metrics: {'train_brier_ensemble': np.float64(0.15735106063965987)}, step: 2007
metrics: {'valid_brier_ensemble': np.float64(0.15546998358451658)}, step: 2007
metrics: {'train_brier_ensemble': np.float64(0.1573726554940911)}, step: 2008
metrics: {'valid_brier_ensemble': np.float64(0.15425624960523532)}, step: 2008
metrics: {'train_brier_ensemble': np.float64(0.1569638269921

WindowsPath('D:/git/code/submissions/submission_xgboost_ensemble_default_features_2025.csv')